In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.9 MB/s eta 0:00:00


In [2]:
import torch
from torch import nn, optim
import re
from torch.utils.data import TensorDataset, DataLoader, random_split
import gensim.downloader as api
from datasets import load_dataset

In [3]:
glove = api.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


In [4]:
# Use GPU instead of CPU for faster results
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
dataset = load_dataset('IMDB')
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [6]:
def tokenize(text):
  text = re.sub(r'<.*?>', ' ', text)  # remove HTML tags
  text = text.lower()
  text = text.replace("'", " '").replace("!", " !").replace("?", " ?")
  return text.split()

In [7]:
class Vocabulary():
  def __init__(self):
    self.wordmap = {
        '<pad>' : 0,
        '<unk>' : 1
    }

  def build(self, sentences):
    for sentence in sentences:
      sentence = tokenize(sentence)
      for word in sentence:
        if word not in self.wordmap.keys():
          self.wordmap[word] = max(self.wordmap.values()) + 1

  def to_indices(self, text):
    indices = []
    for word in tokenize(text):
      if word not in self.wordmap.keys():
        indices.append(1)
      else:
        indices.append(self.wordmap[word])
    return indices

In [8]:
def extract_common_words(limit=10000):
  reviews = dataset['train']['text']
  word_counts = dict()

  for review in reviews:
    review = tokenize(review)
    for word in review:
      if word not in word_counts.keys():
        word_counts[word] = 1
      else:
        word_counts[word] += 1

  sorted_words = sorted(word_counts, key=word_counts.get, reverse=True)

  return sorted_words[:limit]

In [9]:
words = extract_common_words()
vocab = Vocabulary()
vocab.build(words)

In [17]:
class TransformerSentimentModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.embedding = nn.Embedding(num_embeddings=len(vocab.wordmap), embedding_dim=64)
    self.pos_encoding = nn.Embedding(num_embeddings=128, embedding_dim=64) # I reduce the text size to 128 in the file, since it takes ages to train with 256 word long texts.
    self.one_layer = nn.TransformerEncoderLayer(
                                      d_model=64,           # vector size for each word
                                      nhead=2,              # number of attention heads
                                      dim_feedforward=256,  # the "expand" size in feed-forward
                                      dropout=0.1,          # regularization
                                      batch_first=True      # (batch, seq_len, dims) ordering
                                    )
    self.encoder = nn.TransformerEncoder(
        encoder_layer=self.one_layer, # which layer to use
        num_layers=2 # how many times to use
    )
    self.linear = nn.Linear(64, 1)

  def forward(self, x):
    word_vectors = self.embedding(x) # Words -> 64 dimensional vectors
    positions = torch.arange(x.size(1)).to(x.device)  # Create a range array: [0, 1, 2, 3, ..., 255]
    pos_vectors = self.pos_encoding(positions)          # Transform each number 0-255 into a 64 dimensional position vector (256, 64)
    x = word_vectors + pos_vectors # Combine word vectors with their respective position in the text
    x = self.encoder(x) # Pass the words to the multi-attention heads, split, update values, concatenate, normalize, linear (basically everything).
    context = x.mean(dim=1)
    result = torch.sigmoid(self.linear(context))
    return result

In [18]:
def get_review_indices(review):
  indices = vocab.to_indices(review)
  if len(indices) > 128:
    indices = indices[:128]
  else:
    indices += [0] * (128 - len(indices))  # 0 = <pad> index
  return indices

In [25]:
# Prepare data for training/testing
review_tensors_train = []
labels_train = dataset['train']['label']

review_tensors_test = []
labels_test = dataset['test']['label']

for review in dataset['train']['text']:
  review_tensors_train.append(torch.tensor(get_review_indices(review), dtype=torch.long))

for review in dataset['test']['text']:
  review_tensors_test.append(torch.tensor(get_review_indices(review), dtype=torch.long))

X_train = torch.stack(review_tensors_train)
y_train = torch.tensor(labels_train, dtype=torch.float32).reshape(-1, 1)

X_test = torch.stack(review_tensors_test)
y_test = torch.tensor(labels_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_set, val_set = random_split(train_dataset, [20000, 5000])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False)

test_set = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

In [26]:
model = TransformerSentimentModel().to(device)
loss_ft = nn.BCELoss()
opt = optim.Adam(model.parameters(), lr=0.001)

In [27]:
for epoch in range(5):
  train_batch_loss = 0
  val_batch_loss = 0

  model.train()
  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred_train = model(X_batch)
    loss_train = loss_ft(pred_train, y_batch)
    loss_train.backward()
    opt.step()
    opt.zero_grad()

    train_batch_loss += loss_train.item()

  avg_train_batch_loss = train_batch_loss / len(train_loader)


  model.eval()
  with torch.no_grad():
    for X_batch, y_batch in val_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      pred_val = model(X_batch)
      loss_val = loss_ft(pred_val, y_batch)

      val_batch_loss += loss_val.item()

    avg_val_batch_loss = val_batch_loss / len(val_loader)


  print(f'Epoch {epoch}')
  print(f'Training set loss: {avg_train_batch_loss:.4f}')
  print(f'Validation set loss: {avg_val_batch_loss:.4f}')
  print(f'Difference: {(avg_train_batch_loss - avg_val_batch_loss):.4f}')
  print('\n')

Epoch 0
Training set loss: 0.6392
Validation set loss: 0.5955
Difference: 0.0437


Epoch 1
Training set loss: 0.5134
Validation set loss: 0.5203
Difference: -0.0069


Epoch 2
Training set loss: 0.4273
Validation set loss: 0.4951
Difference: -0.0679


Epoch 3
Training set loss: 0.3666
Validation set loss: 0.4824
Difference: -0.1158


Epoch 4
Training set loss: 0.3163
Validation set loss: 0.4872
Difference: -0.1709




In [28]:
total_correct = 0

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = (model(X_batch) > 0.5).float()
        total_correct += (preds == y_batch).sum().item()

acc = total_correct / len(test_set)
print(f"Test accuracy: {(acc * 100):.4f}%")

Test accuracy: 79.1320%
